<a href="https://vigneashpandiyan.github.io/publications/Codes/" target="_blank" rel="noopener noreferrer">
  <img src="https://vigneashpandiyan.github.io/images/Link.png"
       style="max-width: 800px; width: 100%; height: auto;">
</a>

Pytorch Optimizers
==================

Optimizers have a simple job: given gradients of an objective with respect to a set of input parameters, adjust the parameters to reduce the objective (and hopefully, increase accuracy). They do this by modifying each parameter by a small amount in the direction given by the gradient.


Gradient descent just subtracts the gradient
---------
You can apply gradient descent by hand easily by just using `loss.backward()` to compute the gradient of the loss with respect to every parameter `x`, and then apply `x -= learning_rate * x.grad` to nudge `x` in the gradient direction that makes the loss smaller.

Here is an example of applying gradient descent by hand:

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as colors
from matplotlib import pyplot as plt
import matplotlib.ticker as ticker # Import ticker module
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

def plot_progress(bowl, track, losses):
    fig = plt.figure(figsize=(18, 6))

    # 2D plots
    ax1 = fig.add_subplot(131)
    ax2 = fig.add_subplot(132)

    # Use a colormap for the ellipses in 2D plot
    cmap_2d = plt.get_cmap('viridis') # Updated to fix deprecation warning
    norm_2d = colors.Normalize(vmin=0.1, vmax=1.0)

    for i, size in enumerate(torch.linspace(0.1, 1.0, 10)):
        angle = torch.linspace(0, 6.3, 100)
        circle = torch.stack([angle.sin(), angle.cos()])
        ellipse = torch.mm(torch.inverse(bowl), circle) * size
        ax1.plot(ellipse[0,:], ellipse[1,:], color=cmap_2d(norm_2d(size)))

    track_tensor = torch.stack(track)
    losses_tensor = torch.stack(losses)

    ax1.set_title('progress of x (2D)')
    ax1.plot(track_tensor[:,0], track_tensor[:,1], marker='o', color='red', linewidth=2)
    ax1.set_ylim(-1, 1)
    ax1.set_xlim(-1.6, 1.6)
    ax1.set_ylabel('x[1]')
    ax1.set_xlabel('x[0]')

    ax2.set_title('progress of y (2D)')
    ax2.xaxis.set_major_locator(ticker.MaxNLocator(integer=True)) # FIX: Use imported ticker module
    ax2.plot(range(len(losses_tensor)), losses_tensor, marker='o', color='blue', linewidth=2)
    ax2.set_ylabel('objective')
    ax2.set_xlabel('iteration')

    # 3D plot
    ax3 = fig.add_subplot(133, projection='3d')
    ax3.set_title('Objective Function (3D)')

    # Create a grid for the 3D surface
    x_grid = np.linspace(-2.0, 2.0, 50)
    y_grid = np.linspace(-2.0, 2.0, 50)
    X_mesh, Y_mesh = np.meshgrid(x_grid, y_grid)

    bowl_np = bowl.cpu().numpy()
    Z_mesh = np.zeros_like(X_mesh)
    for i in range(X_mesh.shape[0]):
        for j in range(Y_mesh.shape[1]):
            x_vec = np.array([X_mesh[i,j], Y_mesh[i,j]])
            result_vec = np.dot(bowl_np, x_vec)
            Z_mesh[i,j] = np.linalg.norm(result_vec)

    ax3.plot_surface(X_mesh, Y_mesh, Z_mesh, cmap='viridis', alpha=0.6)

    # Plot the optimization track in 3D
    ax3.plot(track_tensor[:,0].cpu().numpy(), track_tensor[:,1].cpu().numpy(), losses_tensor.cpu().numpy(),
             color='red', marker='o', linewidth=3, markersize=5, label='Optimization Path')
    ax3.set_xlabel('x[0]')
    ax3.set_ylabel('x[1]')
    ax3.set_zlabel('Loss')
    ax3.legend()
    ax3.view_init(elev=30, azim=45)

    plt.tight_layout()
    fig.show()

In [ ]:
import torch

x_init = torch.randn(2)
x = x_init.clone()

bowl = torch.tensor([[ 0.4410, -1.0317], [-0.2844, -0.1035]])
track, losses = [], []

for iter in range(21):
    x.requires_grad = True
    loss = torch.mm(bowl, x[:,None]).norm()
    loss.backward()
    with torch.no_grad():
        x = x - 0.1 * x.grad
    track.append(x.detach().clone())
    losses.append(loss.detach())

plot_progress(bowl, track, losses)

Built-in optimization algorithms
------

Pytorch includes several optimization algorithms.

The actual optimization algorithms employ a number of techniques to make the process faster and more robust as repeated steps are taken, by trying to adapt to the shape of the objective surface as it is explored.  The simplest method is SGD-with-momentum, which is implemented in pytorch as `pytorch.optim.SGD`.

Using SGD
---------

To use SGD, you need to calculate your objective and fill in gradients on all the parameters before it can take a step.

  1. Set your parameters (x in this case) to `x.requires_grad = True` so autograd tracks them (line 4).
  2. Create the optimizer and tell it about the parameters to adjust (`[x]` here) (line 5).
  3. In a loop, compute your objective, then call `loss.backward()` to fill in `x.grad` and then `optimizer.step()` to adjust `x` accordingly (lines 11-14).
  
**Remember to zero gradient.** Notice that we use `optimizer.zero_grad()` each time to set x.grad to zero before recomputing gradients; if we do not do this, then the new gradient will be added to the old one.


In [ ]:
import torch

x = x_init.clone()
x.requires_grad = True
optimizer = torch.optim.SGD([x], lr=0.1, momentum=0.5)

bowl = torch.tensor([[ 0.4410, -1.0317], [-0.2844, -0.1035]])
track, losses = [], []

for iter in range(21):
    loss = torch.mm(bowl, x[:,None]).norm()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    track.append(x.detach().clone())
    losses.append(loss.detach())

plot_progress(bowl, track, losses)

### Exercise

1. Set the `x_init` fixed, to the point `torch.tensor([-1.0, 0.0])`.
2. Increase the number of iterations to 101.
3. Experiment with different combinations of `lr` and `momentum` to see which converges to zero `y` best.

Using other optimizers
----------------------

Other optimizers are similar.  Adam is a popular adaptive method that does well without much tuning and can be dropped in to replace plain SGD.

Some other fancy optimizers, such as LBFGS, need to be given an objective function that they can call repeatedly to probe gradients themselves.  Examples can be found elsewhere: for example [hjmshi's LBFGS includes some examples](https://github.com/hjmshi/PyTorch-LBFGS/blob/master/examples/Other/lbfgs_tests.py#L129-L132).

In [ ]:
# The code below uses Adam
x = x_init.clone()
x.requires_grad = True
optimizer = torch.optim.Adam([x], lr=0.1)

track, losses = [], []

for iter in range(21):
    loss = torch.mm(bowl, x[:,None]).norm()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    track.append(x.detach().clone())
    losses.append(loss.detach())

plot_progress(bowl, track, losses)

### RMSprop Optimizer



Here we implement the RMSprop optimizer. We will add a code cell that initializes the variables, sets up the RMSprop optimizer, runs the optimization loop for 101 iterations, and then plot the progress using the provided `plot_progress` function.



In [ ]:
x_init = torch.tensor([-1.0, 0.0])
x = x_init.clone()
x.requires_grad = True

bowl = torch.tensor([[ 0.4410, -1.0317], [-0.2844, -0.1035]])
optimizer = torch.optim.RMSprop([x], lr=0.1, alpha=0.99, eps=1e-08, weight_decay=0, momentum=0, centered=False)

track, losses = [], []

for iter in range(101):
    loss = torch.mm(bowl, x[:,None]).norm()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    track.append(x.detach().clone())
    losses.append(loss.detach())

plot_progress(bowl, track, losses)

### Adagrad Optimizer



In [ ]:
x_init = torch.tensor([-1.0, 0.0])
x = x_init.clone()
x.requires_grad = True

bowl = torch.tensor([[ 0.4410, -1.0317], [-0.2844, -0.1035]])
optimizer = torch.optim.Adagrad([x], lr=0.1, lr_decay=0, weight_decay=0, initial_accumulator_value=0, eps=1e-10)

track, losses = [], []

for iter in range(101):
    loss = torch.mm(bowl, x[:,None]).norm()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    track.append(x.detach().clone())
    losses.append(loss.detach())

plot_progress(bowl, track, losses)

# Optimizer Step (Updates Weights) optimizer.step()

### What happens in `optimizer.step()`?

`optimizer.step()` is the core function that updates the model's parameters (weights and biases) based on the gradients computed during the `loss.backward()` call and the specific optimization algorithm (e.g., SGD, Adam, RMSprop) configured in the optimizer.

Here's a breakdown of what typically occurs:

1.  **Retrieves Gradients**: The optimizer accesses the `grad` attribute of all parameters it is responsible for optimizing. These gradients were previously calculated and stored by PyTorch's autograd engine during the `loss.backward()` call.

2.  **Applies Update Rule**: It then applies the specific update rule defined by the chosen optimization algorithm. For example:
    *   **Stochastic Gradient Descent (SGD)**: `parameter = parameter - learning_rate * parameter.grad`
    *   **SGD with Momentum**: Incorporates a fraction of the previous gradient update to accelerate convergence.
    *   **Adam, RMSprop, Adagrad**: These adaptive optimizers adjust the learning rate for each parameter individually based on past gradients, often by scaling the learning rate by the squared gradients.

3.  **Modifies Parameters In-place**: The `optimizer.step()` method modifies the `data` attribute of the parameters directly. This means the model's weights and biases are updated in-place, moving them in the direction that is expected to reduce the loss function.

**Important Note**: `optimizer.step()` only *uses* the gradients; it does **not** clear them. That's why `optimizer.zero_grad()` is called *before* `loss.backward()` in each training iteration to ensure that gradients don't accumulate across different mini-batches or epochs.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Create a dummy network (single neuron)
# We fix the seed so results are reproducible
torch.manual_seed(42)
model = nn.Linear(1, 1)
optimizer = optim.SGD(model.parameters(), lr=0.1)

# Dummy input and target
data = torch.tensor([[1.0]])
target = torch.tensor([[0.0]])

print(f"{'='*20} INITIAL STATE {'='*20}")
print(f"Weight:   {model.weight.data}")
print(f"Gradient: {model.weight.grad} (None, because we haven't run backward yet)")

# 2. Forward Pass
loss = (model(data) - target).pow(2).mean()

# 3. Backward Pass (Computes Gradients)
loss.backward()

print(f"\n{'='*20} AFTER BACKWARD() {'='*20}")
print(f"Weight:   {model.weight.data} (Unchanged)")
print(f"Gradient: {model.weight.grad} (Calculated!)")

# Store the old weight to compare later
old_weight = model.weight.data.clone()

# 4. Optimizer Step (Updates Weights)
optimizer.step()

print(f"\n{'='*20} AFTER OPTIMIZER.STEP() {'='*20}")
print(f"Weight:   {model.weight.data} (CHANGED!)")
print(f"Gradient: {model.weight.grad} (Still here! Unchanged by step)")

# Check if weight update matches manual calculation:
# New_Weight = Old_Weight - (lr * Gradient)
calculated_weight = old_weight - (0.1 * model.weight.grad)
print(f"\nManual Check: {calculated_weight}")
print(f"Did it match? {torch.allclose(model.weight.data, calculated_weight)}")

# 5. Zero Grad (Clears Gradients)
optimizer.zero_grad()

print(f"\n{'='*20} AFTER ZERO_GRAD() {'='*20}")
print(f"Weight:   {model.weight.data} (Unchanged)")
print(f"Gradient: {model.weight.grad} (Gone/Zeroed)")

## Visualizing the Optimization Process

Let's visualize how the model's single weight and the loss change over several optimization steps. This plot will clearly show the effect of `optimizer.step()` on the weight and how the `loss` is reduced over time.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

# Re-initialize the model and optimizer for a fresh start
torch.manual_seed(42)
model = nn.Linear(1, 1)
optimizer = optim.SGD(model.parameters(), lr=0.1)

# Dummy input and target
data = torch.tensor([[1.0]])
target = torch.tensor([[0.0]])

# Lists to store weight, bias, and loss history
weight_history = []
bias_history = []
loss_history = []

num_iterations = 20 # Run for a few more iterations to see a trend

for i in range(num_iterations):
    # Store current weight and bias before the step
    weight_history.append(model.weight.data.clone().item())
    bias_history.append(model.bias.data.clone().item())

    # Forward pass
    output = model(data)
    loss = (output - target).pow(2).mean()
    loss_history.append(loss.item())

    # Backward pass and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


# Plotting the results
fig, axes = plt.subplots(1, 3, figsize=(21, 5)) # Changed to 3 subplots

# Plot Weight History
axes[0].plot(range(num_iterations), weight_history, marker='o', linestyle='-', color='blue')
axes[0].set_title('Model Weight Over Iterations')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Weight Value')
axes[0].grid(True)

# Plot Loss History
axes[1].plot(range(num_iterations), loss_history, marker='o', linestyle='-', color='red')
axes[1].set_title('Loss Over Iterations')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Loss Value')
axes[1].grid(True)

# Plot Contour of Loss Function with Optimization Path
# Define a grid for weight and bias values
w_vals = np.linspace(-0.5, 1.0, 100)
b_vals = np.linspace(-0.5, 0.5, 100)
W, B = np.meshgrid(w_vals, b_vals)

# Calculate loss for each point on the grid: loss = (w * data + b - target)^2
# Since data=1.0 and target=0.0, loss = (W + B)^2
Z = (W * data.item() + B - target.item())**2

contour = axes[2].contourf(W, B, Z, levels=20, cmap='viridis', alpha=0.8)
fig.colorbar(contour, ax=axes[2], label='Loss Value')
axes[2].plot(weight_history, bias_history, marker='o', linestyle='-', color='red', markersize=5, label='Optimization Path')
axes[2].set_title('Loss Contour with Optimization Path')
axes[2].set_xlabel('Weight')
axes[2].set_ylabel('Bias')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()